In [ ]:
# Esta libreta esta basada en el siguiente repositorio:
# https://github.com/ThiagoLira/ToyDiffusion


import torch
import torch.nn as nn
import torch.optim as optim

import string
import requests
import numpy as np
from PIL import Image
from io import BytesIO

import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm
from operator import mul
from functools import reduce
from IPython.display import Image as IImage
from matplotlib.animation import FuncAnimation, PillowWriter


In [ ]:
# Configuración y parámetros

IMG_SIZE = 256
EPOCHS = 50
training_steps_per_epoch = 40
num_diffusion_timesteps = 30
THRESHOLD = 50 # umbral para binarizar la imagen

beta_start = .0004  # nivel de ruido inicial
beta_end = .02 # nivel de ruido final


In [ ]:

def q_sample_forward(x_start, t, list_bar_alphas, device):
    """
    Generates a noisy version x_t of the clean input x_start according to the
    forward diffusion process q(x_t | x_0) = N(sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I).
    """

    alpha_bar_t = list_bar_alphas[t]

    mean = alpha_bar_t*x_start
    cov = torch.eye(x_start.shape[0]).to(device)
    cov = cov*(1-alpha_bar_t)
    return torch.distributions.MultivariateNormal(loc=mean,covariance_matrix=cov).sample().to(device)


def p_sample_reverse(denoise_model, x_t, t, list_alpha, list_alpha_bar, DATA_SIZE, device):
    """
    Denoising function considering the denoising models tries to model the posterior mean
    """
    alpha_t = list_alpha[t]
    beta_t = 1 - alpha_t
    alpha_bar_t = list_alpha_bar[t]

    mu_theta = denoise_model(x_t,t)

    x_t_before = torch.distributions.MultivariateNormal(loc=mu_theta,covariance_matrix=torch.diag(beta_t.repeat(DATA_SIZE))).sample().to(device)

    return x_t_before


def posterior_q(x_start, x_t, t, list_alpha, list_alpha_bar, device):
    """
    calculate the parameters of the posterior distribution of q
    """
    beta_t = 1 - list_alpha[t]
    alpha_t = list_alpha[t]
    alpha_bar_t = list_alpha_bar[t]
    # alpha_bar_{t-1}
    alpha_bar_t_before = list_alpha_bar[t-1]

    # calculate mu_tilde
    first_term = x_start * torch.sqrt(alpha_bar_t_before) * beta_t / (1 - alpha_bar_t)
    second_term = x_t * torch.sqrt(alpha_t)*(1- alpha_bar_t_before)/ (1 - alpha_bar_t)
    mu_tilde = first_term + second_term

    # beta_t_tilde
    beta_t_tilde = beta_t*(1 - alpha_bar_t_before)/(1 - alpha_bar_t)

    cov = torch.eye(x_start.shape[0]).to(device)*(1-alpha_bar_t)

    return mu_tilde, cov



def position_encoding_init(n_position, d_pos_vec):
    '''
    Init the sinusoid position encoding table
    n_position in num_timesteps and d_pos_vec is the embedding dimension
    '''
    # keep dim 0 for padding token position encoding zero vector
    position_enc = np.array([
        [pos / np.power(10000, 2*i/d_pos_vec) for i in range(d_pos_vec)]
        if pos != 0 else np.zeros(d_pos_vec) for pos in range(n_position)])

    position_enc[1:, 0::2] = np.sin(position_enc[1:, 0::2]) # dim 2i
    position_enc[1:, 1::2] = np.cos(position_enc[1:, 1::2]) # dim 2i+1
    return torch.from_numpy(position_enc).to(torch.float32)


class Denoising(torch.nn.Module):

    def __init__(self, x_dim, num_diffusion_timesteps):
        super(Denoising, self).__init__()

        self.linear1 = torch.nn.Linear(x_dim, x_dim)
        self.emb = position_encoding_init(num_diffusion_timesteps,x_dim)
        self.linear2 = torch.nn.Linear(x_dim, x_dim)
        self.linear3 = torch.nn.Linear(x_dim, x_dim)
        self.relu = torch.nn.ReLU()

    def forward(self, x_input, t):
        emb_t = self.emb[t]
        x = self.linear1(x_input+emb_t)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)
        return x

In [ ]:
def scatter_pixels(img_url):
    """
    Descarga una imagen y devuelve las coordenadas (x, y)
    de los bordes negros detectados.
    """
    response = requests.get(img_url)
    img_bytes = BytesIO(response.content)

    # Abrir imagen y hacerla cuadrada
    img = Image.open(img_bytes).resize((IMG_SIZE, IMG_SIZE)).convert("L")

    def threshold_image(img, threshold):
        """
        Convierte una imagen en blanco y negro según un valor umbral.
        Píxeles con valor < threshold → negros (borde)
        Píxeles con valor >= threshold → blancos (fondo)
        """
        result = img.point(lambda p: 0 if p < threshold else 255)
        return result

    edges = threshold_image(img, THRESHOLD)

    # Obtener los píxeles negros (bordes)
    pixels = edges.load()

    black_pixels = [(x, y) for x in range(IMG_SIZE) for y in range(IMG_SIZE) if pixels[x, y] == 0]

    # Invertir eje Y
    xs = [t[0] for t in black_pixels]
    ys = [IMG_SIZE - t[1] for t in black_pixels]

    return xs, ys


def pack_data(x,y):
    """
    pack 2d data to 1d vector
    """
    one_d_data = []
    for i in range(len(x)):
        one_d_data.append(x[i])
        one_d_data.append(y[i])

    return one_d_data

def unpack_1d_data(one_d_data):
    """
    unpack 1d data to 2d vector
    """
    x = []
    y = []
    for i in range(len(one_d_data)):
        if i%2==0:
            x.append(one_d_data[i])
        else:
            y.append(one_d_data[i])
    return x,y

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
img_url_input = "https://www.infomoney.com.br/wp-content/uploads/2019/06/homer-simpson.jpg"
#img_url_input = "https://wordpressproyectocomputec.wordpress.com/wp-content/uploads/2013/12/tampico.png"
x,y = scatter_pixels(img_url_input)
x = [x/(IMG_SIZE/2) -1 for x in x]
y = [y/(IMG_SIZE/2) -1 for y in y]
#axs = plt.scatter(x,y)

# Guardamos axis para futuras gráficas
ax = sns.scatterplot(x=x, y=y)
y_ax = ax.get_ylim()
x_ax = ax.get_xlim()
axes = (x_ax,y_ax)

print("Número de elementos en x: ", len(x))
print("Número de elementos en y: ", len(y))



In [ ]:
# send data to device
one_d_data = pack_data(x,y) # lista con los puntos en una sola dimensión, en formato x[0],y[0],x[1],y[1]
x_init = torch.tensor(one_d_data).to(torch.float32).to(device)
DATA_SIZE = len(x_init)
print("Tamaño del dataset: ", DATA_SIZE)

In [ ]:
betas = np.linspace(beta_start ** 0.5, beta_end ** 0.5, num_diffusion_timesteps) ** 2    # los 30 niveles de ruido (betas) en un np.array
alphas = 1 - betas

# send parameters to device
betas = torch.tensor(betas).to(torch.float32).to(device)
alphas = torch.tensor(alphas).to(torch.float32).to(device)

# alpha_bar_t is the product of all alpha_ts from 0 to t
list_bar_alphas = [alphas[0]]
for t in range(1,num_diffusion_timesteps):
    list_bar_alphas.append(reduce(mul,alphas[:t]))

list_bar_alphas = torch.cumprod(alphas, axis=0).to(torch.float32).to(device)

TRAINING

In [ ]:
criterion = nn.MSELoss()
denoising_model = Denoising(DATA_SIZE, num_diffusion_timesteps).to(device)
# put embedding layer on 'device' as well, as it is not a pytorch module!
denoising_model.emb = denoising_model.emb.to(device)
optimizer = optim.AdamW(denoising_model.parameters())

In [ ]:

pbar = tqdm(range(EPOCHS))
for epoch in pbar:  # loop over the dataset multiple times

    running_loss = 0.0
    # sample a bunch of timesteps
    Ts = np.random.randint(1,num_diffusion_timesteps, size=training_steps_per_epoch)
    for i, t in enumerate(Ts):
    
        q_t = q_sample_forward(x_init, t, list_bar_alphas, device)

        mu_t, cov_t = posterior_q(x_init, q_t, t, alphas, list_bar_alphas, device)
        sigma_t = cov_t[0][0]
        optimizer.zero_grad()

        mu_theta = denoising_model(q_t , t)
        loss = criterion(mu_t, mu_theta)
        loss.backward()
        optimizer.step()
        running_loss += loss.detach()
        print("Iteration: ", i)
    pbar.set_description('Epoch: {} Loss: {}'.format(epoch, running_loss/training_steps_per_epoch))
print('Finished Training')

REVERSE PROCESS

In [ ]:
data = torch.distributions.MultivariateNormal(loc=torch.zeros(DATA_SIZE),covariance_matrix=torch.eye(DATA_SIZE)).sample().to(device)

for t in tqdm(range(0,num_diffusion_timesteps)):
    data = p_sample_reverse(denoising_model,data,num_diffusion_timesteps-t-1, alphas, list_bar_alphas, DATA_SIZE, device)

In [ ]:
data_cpu = data.detach().cpu().numpy()
x_new, y_new = unpack_1d_data(data_cpu)
sns.scatterplot(x=x_new,y=y_new)

In [ ]:
data = torch.distributions.MultivariateNormal(loc=torch.zeros(DATA_SIZE),covariance_matrix=torch.eye(DATA_SIZE)).sample().to(device)
print(data.shape)
data_cpu = data.detach().cpu().numpy()
data_cpu = unpack_1d_data(data_cpu)
plt.scatter(data_cpu[0],data_cpu[1])

In [ ]:
#!rm output.gif

In [ ]:

fig, ax = plt.subplots(figsize=(6, 6))

def update(frame):
    ax.clear()

    data = torch.distributions.MultivariateNormal(
        loc=torch.zeros(DATA_SIZE),
        covariance_matrix=torch.eye(DATA_SIZE)
    ).sample().to(device)

    data = p_sample_reverse(
        denoising_model, data,
        num_diffusion_timesteps - frame,
        alphas, list_bar_alphas,
        DATA_SIZE, device
    )

    data_plot = data.detach().cpu().numpy()
    x_new, y_new = unpack_1d_data(data_plot)

    sns.scatterplot(ax=ax, x=x_new, y=y_new, color='green', s=10, edgecolor=None)
    ax.set_xlim(axes[0])
    ax.set_ylim(axes[1])
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    ax.text(
        0.98, 0.98, f"Step {frame}/{num_diffusion_timesteps}",
        ha='right', va='top', transform=ax.transAxes,
        fontsize=12, bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2')
    )

anim = FuncAnimation(fig, update, frames=range(1, num_diffusion_timesteps + 1), interval=100, repeat=False)


writer = PillowWriter(fps=24)
anim.save('output.gif', writer=writer, dpi=120)

plt.close(fig)  

print("GIF generado: output.gif")


In [ ]:
IImage(filename="output.gif")